# Dx code-generation benchmark — Colab runner
1. Runtime → Change runtime type → **T4 GPU**.
2. Add your API key in the 🔑 *Secrets* panel (left bar) under the name `ANTHROPIC_API_KEY`
   and enable notebook access.
3. Run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!nvcc --version | tail -1

## 1. Code and dependencies

In [ ]:
!git clone https://github.com/medlemine2206/dx-codegen-bench.git
%cd dx-codegen-bench
!pip -q install -r requirements.txt

## 2. Regenerate the API index from the installed headers, compile-test the docs

In [ ]:
!python tools/extract_api.py
!python tools/dxtools.py check-docs

## 3. Oracle: compile + run the 10 reference solutions (validates every harness, gives reference timings)

In [ ]:
!python eval/run_eval.py --oracle

## 4. Evaluation (≈ 30–60 min for 3 conditions × 10 tasks × 3 samples)

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
MODEL = "claude-haiku-5-5"      # try also "claude-sonnet-5-5"
!python eval/run_eval.py --model {MODEL} --samples 3 --workers 4 --compile-jobs 2

## 5. Report

In [ ]:
!python eval/report.py results/*.jsonl --out results/report
from IPython.display import Image, Markdown, display
import glob
for f in sorted(glob.glob("results/report/*.png")):
    display(Image(f))

## 6. Download everything (commit `results/` to the repo)

In [ ]:
!zip -qr results.zip results && echo ok
from google.colab import files; files.download("results.zip")